# Stacking ensembles

_Classical ML_

**Let several models vote, then train a model to combine the votes.**

Stacking blends several different models into one stronger model.

     First, many base models each make a prediction.

---

This notebook builds a stacking ensemble up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We build a stacking ensemble on a synthetic classification problem in three steps: (1) make the data, (2) define the base models and the stack that combines them, (3) cross-validate each to see whether the stack wins.

### Step 1 — Make a synthetic classification problem

`make_classification` generates 600 examples with 20 features, of which 8 actually carry signal. This gives us a controlled dataset where different model families (trees vs. neighbours) will make different kinds of mistakes — exactly the diversity stacking thrives on.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=600, n_features=20, n_informative=8,
                           random_state=0)

### Step 2 — Define base models and the stacking classifier

The **base models** are a random forest and a k-nearest-neighbours classifier — two very different learners. The `StackingClassifier` trains them, then feeds their out-of-fold predictions into a **final estimator** (logistic regression) that learns how to best combine their votes. The `cv=5` makes those base predictions honest by generating them on held-out folds.

In [ ]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Two diverse base learners.
base = [("rf", RandomForestClassifier(n_estimators=50, random_state=0)),
        ("knn", KNeighborsClassifier(n_neighbors=7))]

# A meta-model learns how to blend the base predictions.
stack = StackingClassifier(estimators=base,
                           final_estimator=LogisticRegression(max_iter=1000),
                           cv=5)

### Step 3 — Cross-validate base models against the stack

We score each base model and the stack with 5-fold cross-validation. If stacking is doing its job, the combined accuracy should match or beat the best individual base model — because the meta-model can lean on whichever base learner is right for each region of the data.

In [ ]:
from sklearn.model_selection import cross_val_score

for name, est in base + [("stack", stack)]:
    acc = cross_val_score(est, X, y, cv=5).mean()
    print("%-6s 5-fold accuracy = %.3f" % (name, acc))

## Visualize the data & results

_On the breast-cancer scan data, does stacking beat each base model alone?_ We repeat the comparison on 569 real tumour scans in three steps: (1) build the models, (2) cross-validate each, (3) plot the accuracies as a bar chart.

### Step 1 — Build the base models and the stack

We use a random forest and a kNN classifier again, but this time kNN is wrapped in a pipeline that **standardises** the 30 features first — distance-based models need comparable scales. The same logistic-regression meta-model sits on top to combine their predictions.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_breast_cancer(return_X_y=True)   # 569 real tumor scans, 30 features

rf = RandomForestClassifier(n_estimators=100, random_state=0)

# kNN needs standardized features, so wrap it in a scaling pipeline.
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))

stack = StackingClassifier(estimators=[("rf", rf), ("knn", knn)],
                           final_estimator=LogisticRegression(max_iter=1000), cv=5)

### Step 2 — Cross-validate each model

We compute the mean 5-fold accuracy for the random forest, the kNN pipeline, and the stacked ensemble. Collecting them in a list lets us line them up for the chart and read off whether the stack edges out its parts.

In [ ]:
from sklearn.model_selection import cross_val_score

labels = ["RandomForest", "kNN", "Stacked"]
accs = [cross_val_score(m, X, y, cv=5).mean() for m in (rf, knn, stack)]

### Step 3 — Plot the accuracies

A bar chart makes the comparison immediate, with the stacked ensemble drawn in green. We zoom the y-axis to the 0.94–0.98 band and label each bar so the small but real differences between the models are visible.

In [ ]:
colors = ["#4ea1ff", "#4ea1ff", "#7ee787"]

plt.bar(labels, accs, color=colors)

# Annotate each bar with its accuracy.
for i, a in enumerate(accs):
    plt.text(i, a, "%.3f" % a, ha="center", va="bottom")

plt.ylim(0.94, 0.98)
plt.title("5-fold accuracy on breast cancer: base models vs stacked ensemble")
plt.ylabel("accuracy")
plt.show()